# Оценка релевантности организаций запросам на Яндекс.Картах с помощью LLM-агента

<img src="https://sun9-65.userapi.com/impg/N4y2cxlL7PauAs82tBNFOUAiNctFICWDy4Mbiw/Jiz1fb7NLWU.jpg?size=1080x1080&quality=95&sign=df2786058624d9ccac3ede4d5d056e2f&type=album" width="500" height="500" />


## Описание и загрузка данных

In [ ]:
import requests

public_url = "https://disk.yandex.ru/d/6d5hFHvpAZjQdw"
api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"

resp = requests.get(api_url, params={"public_key": public_url})
resp.raise_for_status()
download_url = resp.json()["href"] 

dest = "/content/data.jsonl"
with requests.get(download_url, stream=True) as r:
    r.raise_for_status()
    with open(dest, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)


https://downloader.disk.yandex.ru/disk/c4e40269afdee9ca49e9eab0ebe25b18ecbc6af6b47b7840ec075be9647be599/696376ab/K-xrpLEn8irn7zwc4N1sc0c3x5baluqLOdFnR26n5Nz5MY0aF3iGdN-dYxT-VuVMt6gNnqFhuFUCOcSW4gxynQ%3D%3D?uid=0&filename=data_final_for_dls.jsonl&disposition=attachment&hash=9zhUJaGgn780ncxZeBTDjvqMl1xrt3LieF3QJEJ5xW1V4FGhkziXtmNhGd1FBWosq/J6bpmRyOJonT3VoXnDag%3D%3D%3A&limit=0&content_type=text%2Fplain&owner_uid=56285001&fsize=125519961&hid=866f27ccbc30f76938a5f5138e44418c&media_type=text&tknv=v3


In [29]:
import json
import pandas as pd

records = []
with open("/content/data.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f, start=1):
        if i == 2659:      # пропускаем битую строку
            continue
        try:
            obj = json.loads(line)
            records.append(obj)
        except Exception as e:
            print("ещё битая строка:", i, e)

data = pd.DataFrame(records)


In [30]:
data['relevance'].unique()

array([1. , 0. , 0.1])

In [31]:
data['relevance'].value_counts()

,count
relevance,
1.0,15881
0.0,14509
0.1,4703


In [32]:
train_data = data[570:]
eval_data = data[:570]
eval_data = eval_data[eval_data["relevance"] != 0.1]
eval_data

,Text,address,name,normalized_main_rubric_name_ru,permalink,prices_summarized,relevance,reviews_summarized
0,сигары,"Москва, Дубравная улица, 34/29",Tabaccos; Магазин Tabaccos; Табаккос,Магазин табака и курительных принадлежностей,1263329400,None,1.0,"Организация занимается продажей табака, курите..."
1,кальянная спб мероприятия,"Санкт-Петербург, Большой проспект Петроградско...",PioNero; Pionero; Пицца Паста бар; Pio Nero; P...,Кафе,228111266197,PioNero предлагает разнообразные блюда итальян...,0.0,"Организация PioNero — это кафе, бар и ресторан..."
2,Эпиляция,"Московская область, Одинцово, улица Маршала Жу...",MaxiLife; Центр красоты и здоровья MaxiLife; Ц...,Стоматологическая клиника,1247255817,"Стоматологическая клиника, массажный салон и к...",1.0,"Организация занимается стоматологическими, кос..."
4,стиральных машин,"Москва, улица Обручева, 34/63",М.Видео; M Video; M. Видео; M.Видео; Mvideo; М...,Магазин бытовой техники,1074529324,М.Видео предлагает широкий ассортимент бытовой...,1.0,Организация занимается продажей бытовой техник...
5,сеть быстрого питания,"Санкт-Петербург, 1-я Красноармейская улица, 15",Rostic's; KFC; Ресторан быстрого питания KFC,Быстрое питание,1219173871,Rostic's предлагает различные наборы быстрого ...,1.0,"Организация занимается быстрым питанием, предо..."
...,...,...,...,...,...,...,...,...
561,наращивание ресниц,"Саратов, улица имени А.С. Пушкина, 1",Сила; Sila; Beauty brow; Студия бровей Beauty ...,Салон красоты,236976975812,Салон красоты «Сила» предлагает услуги по уход...,1.0,Организация «Сила» занимается предоставлением ...
565,игры,"Москва, Щёлковское шоссе, 79, корп. 1",YouPlay; YouPlay КиберКлуб,Компьютерный клуб,109673025161,YouPlay КиберКлуб предлагает услуги по игре на...,0.0,Организация занимается предоставлением услуг к...
566,домашний интернет в курске что подключить отзы...,"Курск, Садовая улица, 5",Цифровой канал; Digital Channel; DChannel; ЦК;...,Телекоммуникационная компания,1737991898,None,0.0,None
567,гостиница волгодонск сауна номер телефона,"Ростовская область, городской округ Волгодонск...",Поплавок; Poplavok,"База , дом отдыха",147783493467,"Предлагает размещение в различных типах жилья,...",0.0,Организация «Поплавок» предлагает услуги базы ...


In [33]:
eval_data.to_excel("eval_data.xlsx")

# Агент


In [4]:
!pip install -q langchain langgraph langchain_openai langchain_core langchain_community

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [34]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.tools import TavilySearchResults
from langchain_core.tools import tool
import os

In [35]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
search_tool = TavilySearchResults(
    max_results=10,
    include_answer=True,
    include_raw_content=False
)

tools = [search_tool]

In [37]:
llm = ChatOpenAI(
    model="xiaomi/mimo-v2-flash:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0.3,
    max_tokens=128,
    timeout=60,
    max_retries=3).bind_tools(tools)
"""
    extra_body={
        "transformers": ["middle-out"],
        "models": ["xiaomi/mimo-v2-flash:free"],
        "route": "fallback",
        "reasoning": "false"  # Выключает reasoning
    }"""


'\n    extra_body={\n        "transformers": ["middle-out"],\n        "models": ["xiaomi/mimo-v2-flash:free"],\n        "route": "fallback",\n        "reasoning": "false"  # Выключает reasoning\n    }'

In [39]:
def make_example(example: pd.Series, is_train: bool = False):
  ans=f'Запрос для оценки релевантности: {example['Text']}.\n'
  ans+='Информация об организации:\n'
  ans+=f'Адрес: {example['address']}.\n'
  ans+=f'Название: {example['address']}.\n'
  if example['normalized_main_rubric_name_ru']: ans+=f'Описание ассортимента: {example['normalized_main_rubric_name_ru'][:1000]}.\n'
  if example['reviews_summarized'] : ans+=f'Краткое содержание отзывов: {example['reviews_summarized'][:1000]}\n'
  if is_train: ans+=f'Оценка релевантности: {int(example['relevance'])}'
  return ans

Возможно попробовтаь сначала суммаризировать reviews_summarized

In [79]:
train_few_shot_ids = [570, 571, 586, 581, 590, 983]

SYSTEM_PROMPT = f'''
Ты - ассистент, который оценивает, строгую релевантность организаций на картах широким рубричным запросам.\n
Примеры рубричных запросов: "ресторан с верандой", "романтичный джаз-бар".\n
В рубричном запросе важны детали, если деталь не верна, то запрос не релевантен.\n
Возможные ответы:\n
1 - Организация полностью релевантна запросу.\n
0 - Организация НЕ релевантна запросу.\n
Примеры:\n
{make_example(train_data.loc[570],is_train=True)}\n\n
{make_example(train_data.loc[571],is_train=True)}\n\n
{make_example(train_data.loc[586],is_train=True)}\n\n
{make_example(train_data.loc[581],is_train=True)}\n\n
{make_example(train_data.loc[590],is_train=True)}\n\n
{make_example(train_data.loc[983],is_train=True)}\n\n
Оцени релевантность организации запросу одним числом: 0 или 1.\n
Если данных достаточно — дай в ответе одно число: 0 или 1.\n
Добавь 1 предложения пояснения своего выбора.\n
Если данных недостаточно — НЕ ВОЗВРАЩАЙ ЧИСЛО,
а ОБЯЗАТЕЛЬНО вызови Tavily Search с уточняющим запросом.
Пример уточняющего запроса:\n
Страхование автомобиля, КАСКО ОСАГО, застраховать машину в Ингосстрах по адресу Вологодская область,
рабочий посёлок Чагода, улица Кирова, 5Б '''


In [62]:
train_data.loc[580]

,580
Text,Страхование автомобилей
address,"Вологодская область, рабочий посёлок Чагода, у..."
name,"Ингосстрах, офис продаж; Ingosstrakh; Ингосстр..."
normalized_main_rubric_name_ru,Страховая компания
permalink,1765905359
prices_summarized,None
relevance,1.0
reviews_summarized,None


In [80]:
global_debug = []

def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

def gather_data(state: MessagesState):

    messages = state["messages"]
    system_message = (SystemMessage(content=SYSTEM_PROMPT))
    response = llm.invoke([system_message] + messages)
    # debug info
    debug = (json.dumps(response.tool_calls, indent=2,ensure_ascii=False),
    json.dumps(response.content, indent=2,ensure_ascii=False))
    print(debug, flush=True)
    global_debug.append(debug)
    return {"messages" : [response]}

tool_node = ToolNode(tools)
workflow = StateGraph(MessagesState)

workflow.add_node("gather_data", gather_data)
workflow.add_node("tools", tool_node)

workflow.add_conditional_edges("gather_data",should_continue,["tools",END])
workflow.add_edge("tools", "gather_data")

workflow.set_entry_point("gather_data")
graph = workflow.compile()

In [81]:
train_data.loc[574]

,574
Text,балетная школа в санкт петербурге 12лет с прож...
address,"Санкт-Петербург, Невский проспект, 35В"
name,Русская национальная балетная школа Илзе Лиепа...
normalized_main_rubric_name_ru,Школа танцев
permalink,172170646460
prices_summarized,None
relevance,0.0
reviews_summarized,Организация занимается обучением балету и хоре...


In [82]:
prompt = make_example(train_data.loc[574])

input_messages = [HumanMessage(prompt)]
output = graph.invoke({"messages": input_messages})

('[]', '"0\\nОрганизация является школой танцев, но в описании и отзывах нет упоминаний о проживании для учащихся, что является ключевой деталью запроса."')


# Валидация

In [91]:
import numpy as np
from sklearn.metrics import accuracy_score
#измерение точности на n примерах
n = 40
start = 700

predict = []
true = []

for i in range(start, start+n):
  if i in train_few_shot_ids or train_data.loc[i]['relevance'] == 0.1:
    continue
  prompt = make_example(train_data.loc[i])
  input_messages = [HumanMessage(prompt)]
  output = graph.invoke({"messages": input_messages})
  answer = output['messages'][1].content
  predict.append(answer)
  true_answer = train_data.loc[i]['relevance']
  true.append(true_answer)

predict_labels = np.array([int(i[0]) for i in predict ])
true_labels = np.array([int(i) for i in true ])

acc = accuracy_score(predict_labels, true_labels)
print(acc)

('[]', '"0\\nОрганизация находится в Санкт-Петербурге, а не в городе Строитель, что не соответствует запросу."')
('[]', '"0\\nОрганизация является многопрофильным медцентром, а не специализированной частной клиникой для хирургов, и в отзывах нет упоминаний о хирургических услугах."')
('[]', '"0\\nОрганизация является постаматом, что не соответствует запросу на покупку крема для суставов напрямую у производителя."')
('[]', '"1\\nОрганизация предоставляет апартаменты для временного проживания, что соответствует запросу \\"суточный квартира\\"."')
('[]', '"0\\nОрганизация является охранным предприятием (ЧОП), а не специализируется на установке сигнализаций на автомобили, что не соответствует запросу."')
('[]', '"1\\nОрганизация является автосалоном, продающим и обслуживающим автомобили Geely, что полностью соответствует запросу \\"автосалоны geely\\"."')
('[]', '"1\\nОрганизация является муниципальной поликлиникой для взрослых, что соответствует запросу."')
('[]', '"0\\nОрганизация являет

In [93]:
len(predict_labels)

36

In [97]:
# мапа в формате 'TruePredict'
res_map = {'00':0, '10':0, '01':0, '11':0}

for i in range(len(predict_labels)):
    key = str(true_labels[i])+str(predict_labels[i])
    res_map[key]+=1

for k,v in sorted([kv for kv in res_map.items()], key = lambda x: -x[1]):
  print(k,v)


00 15
11 9
10 8
01 4


In [99]:
from sklearn.metrics import recall_score, precision_score, f1_score

precision = precision_score(predict_labels, true_labels)
recall = recall_score(predict_labels, true_labels)
f1 = f1_score(predict_labels, true_labels)
print(precision, recall, f1)

0.5294117647058824 0.6923076923076923 0.6
